# 02. Causal inference: estimate the effect

The first two notebooks prepared the data and measured associations. Now answer the causal question.

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

| Variable | Meaning |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

You work in four steps, and the order matters.

1. **Build the DAG.** Write down what you believe causes what.
2. **Identify.** Ask whether, given that DAG, you can compute the effect from observable data at all.
3. **Estimate.** Compute the number.
4. **Refute.** Stress test the result.

Nearly all the thinking happens in step 1. Steps 2 through 4 run almost mechanically once you fix the assumptions.

**Tutorial path:** 00 Data preparation, then 01 Correlational analysis, then **02 Causal inference**


## 1. The causal estimand

> An **estimand** is the quantity you want to learn, defined before you choose a method. It answers the question "what number do I want?", not the question "how do I compute it?"

Each unit has two potential outcomes.

- **Y(1)** asks whether detection would succeed if the pipeline applied the defense.
- **Y(0)** asks whether detection would succeed under random filtering.

The causal effect for a single unit is the difference `Y(1) - Y(0)`. You can never compute it, because each unit shows you only one of the two.

What you can target is the average across all units, called the average treatment effect, or ATE.

**ATE = E[Y(1) - Y(0)]**

where `E[...]` means the average of.

The outcome is binary, so the ATE gives a difference between two probabilities and reads naturally in percentage points. An ATE of `0.10` says the defense causes an estimated increase of 10 percentage points in the detection success rate, on average.

You can reach the average even though no individual difference is available. With the right assumptions, the treated group stands in for what would have happened to the control group, and the other way round.


## 2. Configure the causal analysis

Point the notebook at the dataset from notebook 00 and the worksheet from notebook 01.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from dowhy import CausalModel

from src.causal_graph_ui import CausalGraphBuilder
from src.causal_tutorial_utils import (
    check_dowhy_version,
    load_analysis_data,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_path": "data/dag_worksheet.csv",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "graph_palette": "colorblind",
        "graph_edge_opacity": 0.35,
        "refuter_simulations": 50,
    }

params = default_params()

print("DoWhy version:", check_dowhy_version("0.14"))
params


## 3. Load the observed data

The model sees the observed treatment, the observed outcome, and the measured covariates. This is exactly what an analyst would hold in a real study.


In [ ]:
(
    analysis_df,
    treatment,
    outcome,
    covariates,
    excluded_covariates,
) = load_analysis_data(params)

print(f"Rows: {len(analysis_df):,}")
print(f"Backdoor-defense prevalence: {analysis_df[treatment].mean():.3f}")
print(f"Observed detection success rate: {analysis_df[outcome].mean():.3f}")
print(f"Covariates available for the graph: {len(covariates)}")
print(f"Excluded constant covariates: {excluded_covariates or 'none'}")

analysis_df.head()


## 4. Review your DAG worksheet

The worksheet records what the data showed, when each variable appeared, and the causal hypotheses you wrote down.

The worksheet is not a causal model. It holds evidence and notes. The DAG you build next is where you commit to assumptions.


In [ ]:
worksheet_path = Path(params["dag_worksheet_path"])

if worksheet_path.exists():
    dag_worksheet = pd.read_csv(worksheet_path)
    display(dag_worksheet)
else:
    dag_worksheet = None
    print(
        "DAG worksheet not found. Run "
        "01_correlational_analysis.ipynb first."
    )


## 5. Build the DAG

> A **DAG**, or directed acyclic graph, draws your causal assumptions. Each variable is a node, and an arrow `A → B` claims that A directly causes B. Leaving an arrow out also makes a claim, namely that no direct effect exists.

The graph starts with the relationship under study.

`treatment → outcome`

That arrow states the hypothesis that applying the defense changes detection success. Everything else you add describes the world around it.

Add an edge only when you can state a plausible mechanism and a consistent time order. Ask two questions before every arrow. Why would A cause B? Did A come first? "They are correlated" answers neither.

Recall the roles from notebook 01.

- A confounder holds its value before treatment and causes both, so it creates bias you need to remove.
- A mediator sits on the path `treatment → X → outcome` and carries part of the effect, so adjusting for it hides what you want to measure.
- A collider has both as causes, and adjusting for it creates bias that was not there.
- A variable can predict the treatment strongly and still have no effect on the outcome.

The editor labels structural roles after you draw the graph, based on the arrows you chose. It does not infer arrows from correlation. That judgment stays yours.


In [ ]:
graph_builder = CausalGraphBuilder(
    data_columns=analysis_df.columns,
    treatment=treatment,
    outcome=outcome,
    covariates=covariates,
    palette=params["graph_palette"],
    edge_opacity=params["graph_edge_opacity"],
)

graph_builder.display()


### Freeze your DAG

When the graph shows assumptions you are willing to defend, click **Use this DAG for analysis**, then run the next cell.

Everything that follows depends on this graph. A different DAG can produce a different estimate from the very same data, which is why you write the assumptions down openly instead of burying them inside a modeling choice.


In [ ]:
analysis_dag = graph_builder.get_frozen_graph()

print(
    f"Using DAG with {analysis_dag.number_of_nodes()} nodes "
    f"and {analysis_dag.number_of_edges()} edges."
)


## 6. Create the DoWhy causal model

DoWhy is a causal inference library. It combines four things: the data, which column holds the treatment, which column holds the outcome, and the DAG you supplied.

The DAG determines the assumptions DoWhy works from, not the correlation matrix. Give it a different graph and you get a different analysis.


In [ ]:
causal_model = CausalModel(
    data=analysis_df,
    treatment=treatment,
    outcome=outcome,
    graph=analysis_dag,
)

print("DoWhy causal model created from the frozen DAG.")


## 7. Identify the causal effect

Identification asks a question that comes before any computation.

> If this DAG is correct, can you write the causal effect in terms of quantities you can actually observe?

The answer does not always come back yes. If the study never measured an important confounder, you may not be able to recover the effect from this data no matter how many rows you have, and no statistical method rescues that. Better to learn it now than after estimating.

DoWhy usually answers with a backdoor adjustment. A backdoor path is a route connecting the treatment and the outcome that does not follow the causal direction, running instead through a common cause, as in `treatment ← X → outcome`. Those paths are what make the raw comparison misleading. Adjusting for the right set of variables blocks them and leaves only the causal effect.

This step settles what to estimate. The next one chooses how.


In [ ]:
identified_estimand = causal_model.identify_effect(
    proceed_when_unidentifiable=False
)

print(identified_estimand)


## 8. Estimate the ATE

This cell uses inverse propensity score weighting.

> A **propensity score** is a unit's probability of receiving the treatment, given its covariates.

The idea runs like this. The treated and control groups are not comparable as they stand, so reweight them until they are. A unit that received a treatment it was unlikely to get counts for more, and a unit that received the expected treatment counts for less. After weighting, the two groups resemble each other on their measured covariates, and the difference in outcomes that remains estimates the effect.

This works only for variables you measured and included. Weighting cannot balance a confounder that never entered the data.

Read the result in context.

- A positive value says the defense increases detection success on average.
- A negative value says it decreases detection success.
- A value near zero says little changes on average, under your assumptions.


In [ ]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    control_value=0,
    treatment_value=1,
    method_params={
        "min_ps_score": 0.05,
        "max_ps_score": 0.95,
        "weighting_scheme": "ips_weight",
    },
)

estimated_ate = float(estimate.value)

print(estimate)
print(f"\nEstimated ATE: {estimated_ate:.4f}")
print(
    "Interpretation: the model estimates a "
    f"{estimated_ate * 100:.1f} percentage-point average change "
    "in detection success when using the backdoor defense rather than "
    "random filtering."
)


## 9. Refute the estimate

Refutation checks rerun the analysis on deliberately altered data and ask whether the estimate reacts the way it should.

They are diagnostics, not proof. Passing them does not show that your DAG is right or that you removed all confounding. It shows only that the estimate survived these particular tests.

### Placebo treatment

This refuter shuffles the treatment column at random, which destroys any real relationship with the outcome. A fake treatment should have no effect, so the estimate should now land near zero. If it does not, the analysis is picking up something other than a treatment effect.


In [ ]:
placebo_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(placebo_refutation)


### Random common cause

This refuter adds a randomly generated variable to the model as an extra common cause. The variable is pure noise, so a sound estimate should barely move. A large shift suggests your estimate is fragile.


In [ ]:
random_common_cause_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(random_common_cause_refutation)


## Final interpretation

The three notebooks answered three different questions.

| Notebook | Question | Answer |
|---|---|---|
| 00 | What did we observe? | One record per unit, with one treatment and one outcome. |
| 01 | What patterns appear? | Associations and imbalances, not causal effects. |
| 02 | What is the causal effect? | A DAG, an identified estimand, an estimated ATE, and robustness checks. |

Your estimate is only as good as the DAG behind it, and that DAG came from your reasoning rather than from the data. So the result worth reporting is not the number alone. It is the number together with its assumptions: which variables you treated as confounders, which as mediators or colliders, and why.

A good closing exercise: name the single assumption that would change your answer most if it turned out to be wrong.

The lesson of the tutorial: deciding which variables to adjust for is a causal reasoning problem, not a correlation ranking exercise.
